# Visión por computador sobre video: los tres proyectos en Colab

### ▶ Para empezar: menú **Entorno de ejecución → Ejecutar todo** (o `Ctrl + F9`)

Tarda unos 5 minutos en total. Luego baja y mira los resultados de cada proyecto.

| proyecto | qué hace |
|---|---|
| 01 · Contador de personas | cuenta entradas, salidas y aforo con YOLO |
| 02 · Lector de placas | lee placas vehiculares con OCR y votación temporal |
| 03 · Nivel de agua | mide el nivel de un río y el riesgo de desborde |

En Colab no se abre una ventana en vivo: cada script procesa el video completo y **al final se muestra el video con las detecciones** aquí mismo.

## 1. Descargar los archivos del curso

Se descargan solos desde Google Drive. No tienes que hacer nada.

In [ ]:
# El profesor pega aquí el ID del archivo vision-clase.zip compartido en Drive
ID_DRIVE = "PEGAR_AQUI_EL_ID"

import os, zipfile, shutil

if ID_DRIVE and ID_DRIVE != "PEGAR_AQUI_EL_ID":
    import gdown
    ruta_zip = "/content/vision-clase.zip"
    gdown.download(id=ID_DRIVE, output=ruta_zip, quiet=False)
else:
    print("No hay ID de Drive configurado: sube tú el archivo vision-clase.zip")
    from google.colab import files
    ruta_zip = next(iter(files.upload()))

shutil.rmtree("/content/tmp_zip", ignore_errors=True)
with zipfile.ZipFile(ruta_zip) as z:
    z.extractall("/content/tmp_zip")

# Busca la carpeta que contiene los tres proyectos, sin importar cómo se comprimió
BASE = None
for raiz, dirs, _ in os.walk("/content/tmp_zip"):
    if "__MACOSX" in raiz:
        continue
    if "01-contador-personas" in dirs:
        BASE = raiz
        break
assert BASE, "No encontré la carpeta 01-contador-personas dentro del zip."

shutil.rmtree("/content/vision-clase", ignore_errors=True)
shutil.move(BASE, "/content/vision-clase")
print("Listo. Proyectos:", sorted(d for d in os.listdir("/content/vision-clase") if d[:2].isdigit()))

## 2. Instalar dependencias (una vez por sesión)

Tarda 1 o 2 minutos. OpenCV y NumPy ya vienen en Colab; se añaden YOLO y el motor de OCR Tesseract.

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install ultralytics pytesseract
!tesseract --version | head -1

## 3. Pantalla en vivo

Colab no puede abrir ventanas. Esta celda hace que el video, con las detecciones encima, se vea **en vivo aquí mismo** mientras se procesa.

Para detener un video antes de que termine, presiona el botón ⏹ de la celda.

In [ ]:
import os, sys, time, runpy, subprocess, base64
import cv2
from IPython.display import display, Image, HTML

def ejecutar_en_vivo(carpeta, comando, ancho=720, fps_pantalla=12):
    """Corre el script y muestra en vivo lo que mandaría a su ventana."""
    os.chdir(os.path.join("/content/vision-clase", carpeta))
    pantalla = display(HTML("<i>Cargando...</i>"), display_id=True)
    ultimo = [0.0]

    def imshow(nombre, img):
        ahora = time.time()
        if ahora - ultimo[0] < 1 / fps_pantalla:
            return
        ultimo[0] = ahora
        h, w = img.shape[:2]
        if w > ancho:
            img = cv2.resize(img, (ancho, int(h * ancho / w)))
        ok, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
        pantalla.update(Image(data=jpg.tobytes()))

    cv2.imshow = imshow
    cv2.waitKey = lambda *a: -1
    cv2.destroyAllWindows = lambda *a: None

    argv = comando.split()
    sys.argv = argv
    try:
        runpy.run_path(argv[0], run_name="__main__")
    except SystemExit as e:
        if e.code not in (None, 0):
            print("Error:", e.code)

def repetir_video(ruta, ancho=720):
    """Vuelve a ver el video grabado (opcional)."""
    web = ruta.replace(".mp4", "_web.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", ruta,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", web], check=True)
    datos = base64.b64encode(open(web, "rb").read()).decode()
    display(HTML(f'<video width="{ancho}" controls><source src="data:video/mp4;base64,{datos}" type="video/mp4"></video>'))

print("OK")

---
## 4. Proyecto 01 · Contador de personas

La primera vez, YOLO puede descargar su modelo (6 MB).

In [ ]:
ejecutar_en_vivo("01-contador-personas",
    "contador.py vtest.avi --motor yolo --speed 1 --end 40 --record salida_colab.mp4 --serie-csv ocupacion.csv")

In [ ]:
import pandas as pd
# repetir_video("salida_colab.mp4")   # quita el # para volver a ver el video
print("Entradas y salidas registradas:")
display(pd.read_csv("conteo.csv").tail(10))

---
## 5. Proyecto 02 · Lector de placas

In [ ]:
ejecutar_en_vivo("02-lector-placas",
    "anpr.py autos.mp4 --formato ecuador --speed 1 --crops recortes --record salida_colab.mp4")

In [ ]:
import pandas as pd
# repetir_video("salida_colab.mp4")   # quita el # para volver a ver el video
print("Placas leídas:")
display(pd.read_csv("placas.csv"))

---
## 6. Proyecto 03 · Nivel de agua

In [ ]:
ejecutar_en_vivo("03-nivel-agua",
    "flood_monitor.py rio_test.mp4 --station 02 --speed 1 --record salida_colab.mp4")

In [ ]:
import pandas as pd
# repetir_video("salida_colab.mp4")   # quita el # para volver a ver el video
df = pd.read_csv("niveles.csv")
display(df.tail(10))

---
## 7. Descargar tus resultados (opcional)

In [ ]:
%cd /content
!zip -qr resultados.zip vision-clase -i "*salida_colab.mp4" "*.csv" "*recortes/*"
# Quita el # de las dos líneas siguientes para descargar
# from google.colab import files
# files.download("/content/resultados.zip")

---
## Para experimentar

Todos los scripts aceptan estas opciones. Cambia el comando y vuelve a ejecutar la celda.

| opción | para qué |
|---|---|
| `--speed 1` | velocidad real (quítalo para ir a toda máquina) |
| `--start 10 --end 40` | procesar solo un tramo del video (en segundos) |
| `--stride 3` | procesar 1 de cada 3 cuadros (más rápido) |
| `--help` | ver todas las opciones |

**¿Quieres usar tu propio video?** Súbelo con el ícono de carpeta 📁 de la izquierda y cambia el nombre del archivo en el comando.

> Nota: el comando va entre comillas dentro de `ejecutar_en_vivo(...)`, sin `python` ni `!` al inicio.

**No olvides leer el `EXPLICACION.md` de cada carpeta.** Ahí se explica cómo funciona cada sistema por dentro.